In [4]:
# ============================================================
# INGEST CIRCUITS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [5]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_LANDING
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

In [ ]:
# -----------------------------------------
# 2. IMPORT HELPER FUNCTIONS
# -----------------------------------------
try:
    add_ingestion_metadata
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    helper_path = "/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb"
    with open(helper_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

✔ Bronze helper functions loaded


In [7]:
# -----------------------------------------
# 3. Define source + target paths
# -----------------------------------------
source_file = f"{BRONZE_LANDING}/circuits.csv"
target_folder = f"{BRONZE_ROOT}/circuits"
target_file = f"{target_folder}/circuits.parquet"

os.makedirs(target_folder, exist_ok=True)

print("Source:", source_file)
print("Target:", target_file)

Source: /Users/manoharazalki/F1-ANALYTICS/data/bronze/landing/circuits.csv
Target: /Users/manoharazalki/F1-ANALYTICS/data/bronze/circuits/circuits.parquet


In [8]:
# -----------------------------------------
# 4. Define schema (same as Databricks)
# -----------------------------------------
circuits_schema = StructType([
    StructField('circuitId', StringType(), True),
    StructField('url', StringType(), True),
    StructField('circuitName', StringType(), True),
    StructField('lat', DoubleType(), True),
    StructField('long', DoubleType(), True),
    StructField('locality', StringType(), True),
    StructField('country', StringType(), True)
])

In [9]:
# -----------------------------------------
# 5. Read CSV
# -----------------------------------------
circuits_df = (
    spark.read
        .format("csv")
        .option("header", True)
        .option("mode", "FAILFAST")
        .schema(circuits_schema)
        .load(source_file)
)

print("✔ Circuits file read successfully")
circuits_df.show(5)

✔ Circuits file read successfully
+---------+--------------------+--------------------+--------+--------+----------+---------+
|circuitId|                 url|         circuitName|     lat|    long|  locality|  country|
+---------+--------------------+--------------------+--------+--------+----------+---------+
|     NULL|https://en.wikipe...|circuit gilles vi...|    45.5|-73.5228|  montreal|   Canada|
|     NULL|https://en.wikipe...|losail internatio...|   25.49| 51.4542|    lusail|    Qatar|
| adelaide|https://en.wikipe...|adelaide street c...|-34.9272| 138.617|  adelaide|Australia|
| ain-diab|https://en.wikipe...|            ain diab| 33.5786| -7.6875|casablanca|  Morocco|
|  aintree|https://en.wikipe...|             aintree| 53.4769|-2.94056| liverpool|       UK|
+---------+--------------------+--------------------+--------+--------+----------+---------+
only showing top 5 rows


In [10]:
# -----------------------------------------
# 6. Add metadata
# -----------------------------------------
circuits_final_df = add_ingestion_metadata(circuits_df, source_file)

print("✔ Metadata added")
circuits_final_df.show(5)

✔ Metadata added
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|circuitId|                 url|         circuitName|     lat|    long|  locality|  country|    ingest_timestamp|         source_file|
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|     NULL|https://en.wikipe...|circuit gilles vi...|    45.5|-73.5228|  montreal|   Canada|2026-06-08 23:07:...|/Users/manoharaza...|
|     NULL|https://en.wikipe...|losail internatio...|   25.49| 51.4542|    lusail|    Qatar|2026-06-08 23:07:...|/Users/manoharaza...|
| adelaide|https://en.wikipe...|adelaide street c...|-34.9272| 138.617|  adelaide|Australia|2026-06-08 23:07:...|/Users/manoharaza...|
| ain-diab|https://en.wikipe...|            ain diab| 33.5786| -7.6875|casablanca|  Morocco|2026-06-08 23:07:...|/Users/manoharaza...|
|  aintree|https://en.wikipe...|      

In [11]:
# -----------------------------------------
# 7. Write to Bronze (Option C)
# -----------------------------------------
(
    circuits_final_df
        .write
        .mode("overwrite")
        .parquet(target_file)
)

print(f"✔ Circuits Bronze written to: {target_file}")

✔ Circuits Bronze written to: /Users/manoharazalki/F1-ANALYTICS/data/bronze/circuits/circuits.parquet


In [12]:
# -----------------------------------------
# 8. Read back for validation
# -----------------------------------------
spark.read.parquet(target_file).show(5)

+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|circuitId|                 url|         circuitName|     lat|    long|  locality|  country|    ingest_timestamp|         source_file|
+---------+--------------------+--------------------+--------+--------+----------+---------+--------------------+--------------------+
|     NULL|https://en.wikipe...|circuit gilles vi...|    45.5|-73.5228|  montreal|   Canada|2026-06-08 23:07:...|/Users/manoharaza...|
|     NULL|https://en.wikipe...|losail internatio...|   25.49| 51.4542|    lusail|    Qatar|2026-06-08 23:07:...|/Users/manoharaza...|
| adelaide|https://en.wikipe...|adelaide street c...|-34.9272| 138.617|  adelaide|Australia|2026-06-08 23:07:...|/Users/manoharaza...|
| ain-diab|https://en.wikipe...|            ain diab| 33.5786| -7.6875|casablanca|  Morocco|2026-06-08 23:07:...|/Users/manoharaza...|
|  aintree|https://en.wikipe...|             aintree| 5